In [1]:
import tensorflow.keras
import tensorflow as tf
from tensorflow.keras.layers import Conv2D, Conv3D, Flatten, Dense, Reshape, BatchNormalization, Activation, MaxPooling3D, MaxPooling2D, Bidirectional
from tensorflow.keras.layers import InputLayer, Dropout, Input
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.utils import to_categorical

from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report, cohen_kappa_score 

from operator import truediv

from plotly.offline import init_notebook_mode
from tensorflow.keras.optimizers import SGD
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.models import load_model

import numpy as np
import scipy.io as sio
from PIL import Image
import matplotlib.pyplot as plt
import geopandas as gdp
import pandas as pd
import sys
import os
import spectral
import argparse
import random
import time
from sklearn.metrics import f1_score
from scipy.ndimage import generic_filter
init_notebook_mode(connected=True)
%matplotlib inline

# %matplotlib notebook

# path='H:/3.6 all data/guangdong/LUtxt/'
# files_1 = os.listdir(path)
# print(files_1)
# print(files_1[0])

In [2]:
# os.environ['CUDA_VISIBLE_DEVICES'] = '6'

In [3]:
import copy

In [4]:
import tensorflow as tf
print(tf.test.is_gpu_available())
# 检查是否使用GPU
if tf.test.gpu_device_name():
    print('GPU已安装')
else:
    print('GPU未安装')
    
import sys
print(sys.version)
import os
print(os.environ.get("CUDA_VISIBLE_DEVICES"))

Instructions for updating:
Use `tf.config.list_physical_devices('GPU')` instead.
True
GPU已安装
3.8.18 | packaged by conda-forge | (default, Dec 23 2023, 17:17:17) [MSC v.1929 64 bit (AMD64)]
7


In [5]:
#测试一下读取数据顺序
pa='D:/yangwei/zhangyu/A-N-CA/LU/'
fil=os.listdir(pa)
for i in range(5):
    print(fil[i])
print('lu....end...')
# pa='D:/yangwei/zhangyu/A-N-CA/driving/2010driving/'
# fil=os.listdir(pa)
# for i in range(8):
#     print(fil[i])
# print('10driving....end...')
# pa='D:/yangwei/zhangyu/A-N-CA/driving/2015driving/'
# fil=os.listdir(pa)
# for i in range(8):
#     print(fil[i])
# print('15driving....end...')

gd-lu-00-txt.txt
gd-lu-05-txt.txt
gd-lu-10-txt.txt
gd-lu-15-txt.txt
gd-lu-20-txt.txt
lu....end...


In [6]:
#定义基本参数
paths=['2005driving-2/','2010driving-2/','2015driving-2/','2020driving-2/']
year = 3
channel=13
windowSize=11
test_ratio = 0.7

In [7]:
def loadData(tel = 0): #例：驱动：00，05 LU：05，10
    pathLU='D:/yangwei/zhangyu/A-N-CA/LU/'
    pathDV='D:/yangwei/zhangyu/A-N-CA/driving/'
    A = np.zeros((605,755))
    B = np.zeros((605,755))
    C = np.zeros((605,755,3,9))
    
    driv1=pathDV+paths[tel]
    filesdriv1=os.listdir(driv1)
    driv2=pathDV+paths[1+tel]
    filesdriv2=os.listdir(driv2)
    driv3=pathDV+paths[2+tel]
    filesdriv3=os.listdir(driv3)
    for i in range(9):
        C[:,:,0,i] = np.loadtxt(driv1+filesdriv1[i])
        C[:,:,1,i] = np.loadtxt(driv2+filesdriv2[i])
        C[:,:,2,i] = np.loadtxt(driv2+filesdriv2[i])
        print('load...wait...'+str(i))
    files=os.listdir(pathLU)
    A = np.loadtxt(pathLU+files[2+tel])
    B = np.loadtxt(pathLU+files[3+tel])
    return A, B, C

In [8]:
A2, B2, C2 = loadData(tel=1)#A2:2015LU  B2：2020LU  C2：2010-2020 driving
A2 = A2-1
B2 = B2-1
true_2020 = copy.deepcopy(B2) 

load...wait...0
load...wait...1
load...wait...2
load...wait...3
load...wait...4
load...wait...5
load...wait...6
load...wait...7
load...wait...8


In [9]:
def padWithZeros(X, margin=5):#605,755,8   5 
    newX = np.zeros((X.shape[0] + 2 * margin, X.shape[1] + 2* margin, 3, 9))#605,755,8    
    x_offset = margin
    y_offset = margin
    newX[x_offset:X.shape[0] + x_offset, y_offset:X.shape[1] + y_offset, :, : ] = X
    return newX # 615,765 ,3,8

In [10]:
def createImageCubes1(A, B, C): #output： A1, B1, C1, T
    margin = int((windowSize - 1) / 2) # 5
    zeroPaddedX = padWithZeros(C, margin=margin)#615,765,8 LU图扩大5像素
    patchesData = np.zeros((A.shape[0] * A.shape[1], windowSize, windowSize, C.shape[2],C.shape[3]))#456775,11,11,8
    patchesLabelsi = np.zeros((A.shape[0] * A.shape[1]))#456775
    patchesLabelsf = np.zeros((A.shape[0] * A.shape[1]))#456775 
    T = np.zeros((A.shape[0] * A.shape[1],2)) #定标
    patchIndex = 0
    for r in range(margin, zeroPaddedX.shape[0] - margin):#5,610
        for c in range(margin, zeroPaddedX.shape[1] - margin):#5,760
            patch = zeroPaddedX[r - margin:r + margin + 1, c - margin:c + margin + 1,:,:]
            patchesData[patchIndex, :, :, :, :] = patch      
            patchesLabelsi[patchIndex] = A[r-margin, c-margin]
            patchesLabelsf[patchIndex] = B[r-margin, c-margin]
            T[patchIndex, :] = [r - margin, c - margin]
            patchIndex = patchIndex + 1
    print('wait...patchindex＝:'+str(patchIndex))
    patchesData = patchesData[patchesLabelsi>-1,:,:,:,:] #样本有用
    T = T[patchesLabelsi>-1,:]
    patchesLabelsf = patchesLabelsf[patchesLabelsi>-1]
    patchesLabelsi = patchesLabelsi[patchesLabelsi>-1]
    t = 0
    for i in range (patchesLabelsi.shape[0]):
        if int(patchesLabelsf[i])>-1 and int(patchesLabelsi[i])>-1 :
            patchesData[t, :, :, :, :] = patchesData[i, :, :, :, :]
            patchesLabelsi[t] = patchesLabelsi[i]
            patchesLabelsf[t] = patchesLabelsf[i]
            T[t,:] = T[i,:]  
            t = t+1
    patchesDataC = patchesData[0:t, :, :, :, :]
    patchesLabelsA = patchesLabelsi[0:t] #f 或者i，标签 10年
    patchesLabelsB = patchesLabelsf[0:t] #f 或者i，标签 15年
    T_T = T[0:t,:]
    # 遍历数组中的每个元素
    for i in range(patchesDataC.shape[0]):  # 遍历样本 (t)
        for k in range(patchesDataC.shape[3]):  # 遍历新增维度 (2)
            for j in range(patchesDataC.shape[4]):  # 遍历通道 (9)
                window = patchesDataC[i, :, :, k, j]  # 取出窗口
                if -9999 in window:
                    # 计算窗口中非-9999的平均值
                    valid_values = window[window != -9999]
    
                    if valid_values.size > 0:  # 检查是否存在正常值
                        mean_val = np.mean(valid_values)  # 存在正常值，计算平均值
                    else:
                        mean_val = 0  # 不存在正常值，设置为 0
    
                    # 替换-9999值
                    window[window == -9999] = mean_val
                    patchesDataC[i, :, :, k, j] = window  # 替换回原数据

    return patchesLabelsA, patchesLabelsB, patchesDataC, T_T

In [11]:
#2.windows裁剪大小，实际 存在的样本和标签
A12, B12, C12, T2 = createImageCubes1(A2, B2, C2)  #A12:2015LU实际数据样本集  B12：2020LU实际数据样本集 C12：对应2005-2010-2015 Driving T2:真实数据在LU中的位置
print(A12.shape, B12.shape, C12.shape, T2.shape)

wait...patchindex＝:456775
(176011,) (176011,) (176011, 11, 11, 3, 9) (176011, 2)


In [12]:

T2 = T2.astype(int)

In [13]:
# 找到 C1 中的 NaN 值位置
nan_indices = np.isnan(C12)
# 将 NaN 值位置的值填充为 0
C12[nan_indices] = 0

In [14]:
model_test_3DCNN_1 = load_model('C:\\Users\\yangwei\\Desktop\\other-model\\3DCNN-CA\\3DCNN-2.hdf5')

In [15]:
Y_pred_test_3DCNN_2 = model_test_3DCNN_1.predict(C12) #预测预测集

y_pred_test_3DCNN_2 = np.argmax(Y_pred_test_3DCNN_2, axis=1)#预测结果最大的

In [16]:
# 初始化 P0 为 -10000，形状为 (605, 755, 6)
P4 = np.full((605, 755, 6), -10000, dtype=np.float32)
# 使用 T 中的索引，将 Y_pred_test_3DCNN_1[:, :6] 的值赋给 P0 对应位置
P4[T2[:, 0], T2[:, 1], :] = Y_pred_test_3DCNN_2[:, :6]
print(P4.shape)

(605, 755, 6)


In [17]:
#开始NCA预先训练！！！

In [18]:
# 邻域统计函数
def compute_neighborhood_stats2(A):
    stats = np.zeros((A.shape[0], A.shape[1], 6))-10000  # 每个栅格点的邻域统计信息
    for land_type in range(6):
        mask = (A == land_type).astype(np.float32)
        stats[:, :, land_type] = tf.nn.conv2d(
            mask[np.newaxis, :, :, np.newaxis],
            filters=np.ones((3, 3, 1, 1)), 
            strides=1, padding="SAME").numpy().squeeze() / 9
    return stats

In [19]:
# 创建每个选取点及其邻域的输入数据 A0
def create_A0(A, P, T):
    A0 = []
    neighborhood_stats = compute_neighborhood_stats2(A)
    
    for i, (x, y) in enumerate(T):
        # 初始化 11x11 的邻域区域，并处理边界超出区域
        neighborhood_A = np.full((11, 11), -10000, dtype=A.dtype)
        neighborhood_P = np.full((11, 11, 6), -10000, dtype=P.dtype)
        neighborhood_stats_patch = np.full((11, 11, 6), -10000, dtype=neighborhood_stats.dtype)
        
        # 确定邻域的有效范围，避免超出边界
        x_min = max(0, x - 5)
        x_max = min(A.shape[0], x + 6)
        y_min = max(0, y - 5)
        y_max = min(A.shape[1], y + 6)
        
        # 计算在 11x11 网格中的相对位置
        neighborhood_A[(x_min - x + 5):(x_max - x + 5), (y_min - y + 5):(y_max - y + 5)] = A[x_min:x_max, y_min:y_max]
        neighborhood_P[(x_min - x + 5):(x_max - x + 5), (y_min - y + 5):(y_max - y + 5), :] = P[x_min:x_max, y_min:y_max, :]
        neighborhood_stats_patch[(x_min - x + 5):(x_max - x + 5), (y_min - y + 5):(y_max - y + 5), :] = neighborhood_stats[x_min:x_max, y_min:y_max, :]
        
        # 初始化 11x11x8 特征矩阵，并填充对应通道
        neighborhood_features = np.zeros((11, 11, 13))
        neighborhood_features[:, :, 0] = neighborhood_A
        neighborhood_features[:, :, 1:7] = neighborhood_P
        neighborhood_features[:, :, 7:13] = neighborhood_stats_patch

        # 检查是否存在 -10000，如果存在，则用非 -10000 的均值替换
        mask = (neighborhood_features != -10000)
        for channe in range(13):
            valid_values = neighborhood_features[:, :, channe][mask[:, :, channe]]
            if valid_values.size > 0:
                mean_value = valid_values.mean()
                neighborhood_features[:, :, channe][~mask[:, :, channe]] = mean_value
            else:
                # 如果整个区域为 -10000，则替换为 0
                neighborhood_features[:, :, channe].fill(0)
        
        A0.append(neighborhood_features)
    
    return np.array(A0)

In [20]:
def calculate_fom(A, B, C):
    # 计算变化的情况，不包括真实和预测都不变的情况
    actual_change = tf.not_equal(A, B)
    predicted_change = tf.not_equal(A, C)

    # 真实变化与预测变化都发生的区域
    correct_prediction = tf.logical_and(actual_change, tf.equal(B, C))

    # 计算分子：正确预测的变化样本数
    numerator = tf.reduce_sum(tf.cast(correct_prediction, tf.float32))

    # 计算分母：所有实际变化的样本数加上所有错误预测的变化样本数
    denominator = tf.reduce_sum(tf.cast(actual_change, tf.float32)) + tf.reduce_sum(tf.cast(tf.logical_and(tf.logical_not(actual_change), predicted_change), tf.float32))

    # 计算FOM
    fom = numerator / denominator
    return fom.numpy()

In [22]:
model_CNN = load_model('C:\\Users\\yangwei\\Desktop\\other-model\\Conv-7\\Best-CNN-conv-7.h5')

In [23]:
#使用预训练的cnn迭代5次，模拟2020
A_state2 = copy.deepcopy(true_2020)
for year in range(5):
    print(f"yxz...simulate_land_use...: {year}")
    # 获取当前状态输入，A0 形状为 (160000, 11, 11, 8)
    A0 = create_A0(A_state2, P4, T2)
    print(f"yxz...A0...: {year}")
    # 前向传播，land_type_logits 形状为 (160000, 6)
    land_type_logits = model_CNN(A0, training=False)
    print(f"yxz...model...: {year}")
    # 获取预测类别，形状为 (160000,)
    predicted_land_types = tf.argmax(land_type_logits, axis=-1)

    # 生成随机数数组 Asy
    Asy = tf.random.uniform(shape=(len(T2),), minval=0.0, maxval=1.0)  # 随机生成范围 [0, 1)
    # 更新土地利用图状态，将预测的类别应用到 A_state 中
    for i, (x, y) in enumerate(T2):
        if Asy[i] > 0.5:  # 判断随机数是否大于 0.5
            A_state2[x, y] = predicted_land_types[i]
        
print(f"yxz...2015-2020...end...")

yxz...simulate_land_use...: 0
yxz...A0...: 0
yxz...model...: 0
yxz...simulate_land_use...: 1
yxz...A0...: 1
yxz...model...: 1
yxz...simulate_land_use...: 2
yxz...A0...: 2
yxz...model...: 2
yxz...simulate_land_use...: 3
yxz...A0...: 3
yxz...model...: 3
yxz...simulate_land_use...: 4
yxz...A0...: 4
yxz...model...: 4
yxz...2015-2020...end...


In [24]:
A_staten = A_state2+1

In [25]:
np.savetxt('C:\\Users\\yangwei\\Desktop\\txt-3\\predict-conv-7-2025.txt', A_staten, fmt='%d')